In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
trainX = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
trainY = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
trainX.head()

In [ ]:
# rename all columns

cols = ['Column_' + str(i).zfill(1) for i in range(40)]
trainX.columns = cols
test.columns = cols

In [ ]:
trainX.head()

In [ ]:
trainX.describe()

In [ ]:
# check if dataset contains null values

trainX.isnull().sum(axis=0)

In [ ]:
trainY.columns=['label']
train= pd.concat([trainX, trainY], axis=1)
train.describe()

In [ ]:
# let's see distribution of labels

sns.countplot(train.label).set_title('Count of label values', fontsize=15)
plt.show()
print(train.label.value_counts())

In [ ]:
# Handeling outliers

def remove_outlier(df):
    for col in df.columns:
        if (((df[col].dtype)=='float64') | ((df[col].dtype)=='int64')):
            percentiles = df[col].quantile([0.01,0.99]).values
            df[col][df[col] <= percentiles[0]] = percentiles[0]
            df[col][df[col] >= percentiles[1]] = percentiles[1]
        else:
            df[col]=df[col]
    return df



In [ ]:
train_new=remove_outlier(train)

In [ ]:
train_new.describe()

In [ ]:
# import sciKit learn libraries

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics         import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition   import PCA

from sklearn.linear_model    import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes     import GaussianNB, BernoulliNB
from sklearn.svm             import SVC
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

In [ ]:
def classification_(X,y): 
    xgbc=XGBClassifier()
    g=GaussianNB()
    b=BernoulliNB()
    k=KNeighborsClassifier()
    svc=SVC()
    d=DecisionTreeClassifier()
    log=LogisticRegression()
    gbc=GradientBoostingClassifier()
    rf=RandomForestClassifier()
    ab=AdaBoostClassifier()
    
    algos=[xgbc,g,b,k,svc,d,log,gbc,rf,ab]
    algos_name=['XGBClassifier','Gaussian','Bernoulli','KNeighbors','SVC','DecisionTree','LogisticReg','GradientBoosting','RandomForest','AdaBoost']

    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=101)

    accuracy = []
    precision = []
    recall = []
    f1 = []
   
    result=pd.DataFrame(columns=['AccuracyScore','PrecisionScore','RecallScore','f1_Score'],index=algos_name)
    
    for i in algos:
        predict=i.fit(X_train,y_train).predict(X_test)
        accuracy.append(accuracy_score(y_test,predict))
        precision.append(precision_score(y_test,predict))
        recall.append(recall_score(y_test,predict))
        f1.append(f1_score(y_test,predict))
      
    result.AccuracyScore=accuracy
    result.PrecisionScore=precision
    result.RecallScore=recall
    result.f1_Score=f1
    
    return result.sort_values('AccuracyScore',ascending=False)

In [ ]:
df_train=pd.read_csv('../input/data-science-london-scikit-learn/train.csv', header=None)
labels = pd.read_csv('../input/data-science-london-scikit-learn/trainLabels.csv', header=None)
df_test=pd.read_csv('../input/data-science-london-scikit-learn/test.csv', header=None)

In [ ]:
df_train.shape, labels.shape, df_test.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
train=scaler.fit_transform(df_train)
test=scaler.transform(df_test)

In [ ]:
classification_(train, labels)

In [ ]:
from xgboost import XGBClassifier
xgbc=XGBClassifier()
predictions=xgbc.fit(train,labels).predict(test)

In [ ]:
Result = pd.DataFrame()
Result['Id']=list(range(1,9001))
Result['Solution']=predictions
Result.to_csv('result.csv', index=False)

In [ ]:
Result